## 08-statement-logistic

Ответ: 

Без регуляризации: AUC = 0.717619

C регуляризацией (C=10.0): AUC = 0.935524

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


data = pd.read_csv('data-logistic.csv', header=None)
X = data.iloc[:, 1:3].values   # первые два столбца — признаки
y = data.iloc[:, 0].values    # третий столбец — целевая переменная {-1, 1}

# Параметры градиентного спуска
k = 0.1               # шаг
tol = 1e-5            # критерий остановки
max_iter = 10000      # максимальное число итераций
C_reg = 10.0          # коэффициент регуляризации

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def compute_auc(w, X, y):
    """Вычисляет AUC-ROC по весам w и данным X, y."""
    scores = sigmoid(X @ w)   # вероятности принадлежности к классу +1
    return roc_auc_score(y, scores)

def gradient_descent(X, y, C, k, tol, max_iter):
    """
    Реализует градиентный спуск для логистической регрессии с L2-регуляризацией.
    Если C = 0, регуляризация отключается.
    Возвращает вектор весов w.
    """
    l = len(y)                     # количество объектов
    w = np.zeros(X.shape[1])       # начальное приближение (0, 0)
    
    for _ in range(max_iter):
        # Вычисляем линейную комбинацию w1*x1 + w2*x2 для всех объектов
        linear = X @ w

        arg = -y * linear
 
        one_minus_sigma = 1.0 - sigmoid(arg)
        

        grad = (1.0 / l) * ( (y * one_minus_sigma).reshape(-1,1) * X ).sum(axis=0)

        if C > 0:
            grad += C * w
        
        # Шаг градиентного спуска
        w_new = w + k * grad       # w := w + k * ( -производная? ) – тут важно!
        
        grad_no_reg = (1.0 / l) * ( (y * one_minus_sigma).reshape(-1,1) * X ).sum(axis=0)
        if C > 0:
            w_new = (1 - k*C) * w + k * grad_no_reg
        else:
            w_new = w + k * grad_no_reg
        
        # Проверка сходимости
        if np.linalg.norm(w_new - w) < tol:
            return w_new
        w = w_new
    
    return w

# 1. Обучение без регуляризации (C = 0)
w_no_reg = gradient_descent(X, y, C=0.0, k=k, tol=tol, max_iter=max_iter)
auc_no_reg = compute_auc(w_no_reg, X, y)
print(f"Без регуляризации: AUC = {auc_no_reg:.6f}")

# 2. Обучение с регуляризацией C = 10
w_reg = gradient_descent(X, y, C=C_reg, k=k, tol=tol, max_iter=max_iter)
auc_reg = compute_auc(w_reg, X, y)
print(f"C регуляризацией (C={C_reg}): AUC = {auc_reg:.6f}")

# Ответ в требуемом формате (два числа с тремя знаками после точки)
print(f"{auc_no_reg:.3f} {auc_reg:.3f}")

C:\Users\Даниил\AppData\Local\Temp\ipykernel_4940\1463892843.py:17: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-z))


Без регуляризации: AUC = 0.717619
C регуляризацией (C=10.0): AUC = 0.935524
0.718 0.936
